# 04 | Module Composition and Tensor Contraction

This notebook builds larger models from leaf modules and introduces `concat`, `hermit`, `bra`, trace operations, and the two registered contraction strategies.


In [1]:
from pathlib import Path
import sys

# This works whether Jupyter starts in the repository root or in notebooks/.
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "tneq_qc").is_dir() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/yuch3n/Documents/Code/Github/tneq-qc


In [2]:
from tneq_qc import BackendFactory, EngineCommon, QCTN
from tneq_qc.modules.small import MPS, State, MeasureMatrix

backend = BackendFactory.create_backend("pytorch", device="cpu", dtype="float32")


## 1. Leaf modules

- `State` is a product-state ket, initialized to `|0⟩` on every qubit.
- `MPS` is a tensor-network component that may be fixed or trainable.
- `MeasureMatrix` provides one square operator per qubit.

Constructors establish topology; `auto_init()` creates core weights.


In [3]:
state = State(3, phys_dim=2, backend=backend).auto_init()
mps = MPS(3, bond_dim=2, phys_dim=2, backend=backend).auto_init(orthogonal=True)
mx = MeasureMatrix(3, phys_dim=2, backend=backend).auto_init(orthogonal=True)

print("State:", state)
print("MPS:", mps)
print("Measurement:", mx)


State: QCTN(nqubits=3, cores=[a(2,), b(2,), c(2,)])
MPS: QCTN(nqubits=3, cores=[a(2, 2, 2, 2), b(2, 2, 2, 2)])
Measurement: QCTN(nqubits=3, cores=[a(2, 2), b(2, 2), c(2, 2)])


## 2. Horizontal composition with `QCTN.concat`

Give every segment a prefix. Core symbols are reassigned during concatenation, but readable names such as `state.a`, `tn.a`, and `mx.a` remain available.


In [4]:
combined = QCTN.concat([
    ("state", state),
    ("tn", mps),
    ("mx", mx),
])

print(combined)
print("\nCombined graph:")
print(combined.to_graph_string())
print("\nReadable names:")
print(list(combined.core_names.values()))


QCTN(nqubits=3, cores=[state.a(2,), state.b(2,), state.c(2,), tn.a(2, 2, 2, 2), tn.b(2, 2, 2, 2), mx.a(2, 2), mx.b(2, 2), mx.c(2, 2)])

Combined graph:
-a-2-d-2-f-2-
-b-2-d-2-e-2-g-2-
-c-2-e-2-h-2-

Readable names:
['state.a', 'state.b', 'state.c', 'tn.a', 'tn.b', 'mx.a', 'mx.b', 'mx.c']


## 3. The closed five-segment structure

A Born-style expectation uses:

```text
state + tn + mx + tn_h + state_bra
```

`tn.hermit()` reverses and conjugates the trainable network. `state.bra()` creates the matching closing boundary. With every external edge closed, the output is a scalar, or one scalar per batch item.


In [5]:
# Initialize measurement cores as identities for a structural check.
for info in mx.adjacency_table:
    mx[info["core_name"]] = backend.eye(info["input_dim"])

closed = QCTN.concat([
    ("state", state),
    ("tn", mps),
    ("mx", mx),
    ("tn_h", mps.hermit()),
    ("state_t", state.bra()),
])

row_engine = EngineCommon(backend=backend, strategy="row_priority")
print("Closed-network result:", row_engine.contract(closed))


[Compiler] Strategy candidates: ['row_priority'], Testing 1 strategies...
  [row_priority] Compatibility: True
  [row_priority] Estimated cost: 5.00e+05 FLOPs
[Compiler] Selected strategy: row_priority (cost: 5.00e+05)
Closed-network result: TNTensor(shape=torch.Size([]), scale=1)


## 4. Contraction strategies

The currently registered built-in strategies are:

- `row_priority`: contracts qubit rows progressively; recommended for batched BornMachine and larger networks;
- `einsum_default`: creates a global Einstein summation expression and lets `opt_einsum` plan its path.

A list of strategy names may be supplied when you want the compiler to compare compatible candidates.


In [6]:
from tneq_qc import get_registered_contraction_strategies

print(get_registered_contraction_strategies().keys())

einsum_engine = EngineCommon(backend=backend, strategy="einsum_default")
print("Einsum result:", einsum_engine.contract(closed))


dict_keys(['einsum_default', 'row_priority'])
[Compiler] Strategy candidates: ['einsum_default'], Testing 1 strategies...
  [einsum_default] Compatibility: True
einsum_eq for cost estimation: a,b,c,abde,ecfg,dh,fi,gj,hklm,ijkn,l,m,n,o,p,q,rsto,utqp,vs,wr,xu,yzwv,ABxy,z,B,A->
tensor_shapes for cost estimation: [torch.Size([2]), torch.Size([2]), torch.Size([2]), torch.Size([2, 2, 2, 2]), torch.Size([2, 2, 2, 2]), torch.Size([2, 2]), torch.Size([2, 2]), torch.Size([2, 2]), torch.Size([2, 2, 2, 2]), torch.Size([2, 2, 2, 2]), torch.Size([2]), torch.Size([2]), torch.Size([2]), torch.Size([2]), torch.Size([2]), torch.Size([2]), torch.Size([2, 2, 2, 2]), torch.Size([2, 2, 2, 2]), torch.Size([2, 2]), torch.Size([2, 2]), torch.Size([2, 2]), torch.Size([2, 2, 2, 2]), torch.Size([2, 2, 2, 2]), torch.Size([2]), torch.Size([2]), torch.Size([2])]
  [einsum_default] Estimated cost: 1.00e+00 FLOPs
einsum_equation a,b,c,abde,ecfg,dh,fi,gj,hklm,ijkn,l,m,n,o,p,q,rsto,utqp,vs,wr,xu,yzwv,ABxy,z,B,A->
tensor

## 5. Trace operations

`set_trace()` closes selected qubit boundaries. Trace semantics are primarily implemented by `row_priority` in the current version.


In [7]:
left = MPS(3, bond_dim=2, phys_dim=2, backend=backend).auto_init(orthogonal=True)
overlap = QCTN.concat([("left", left), ("right", left.hermit())])
overlap.set_trace("all")

print("Full-trace result:", row_engine.contract(overlap))


[Compiler] Strategy candidates: ['row_priority'], Testing 1 strategies...
  [row_priority] Compatibility: True
  [row_priority] Estimated cost: 5.00e+05 FLOPs
[Compiler] Selected strategy: row_priority (cost: 5.00e+05)
Full-trace result: TNTensor(shape=torch.Size([]), scale=1)


## 6. Compilation cache

The compiled function is cached on a QCTN instance. Replacing core values with tensors of the same shape usually reuses the cache. Changing the graph, trace configuration, or tensor shapes requires recompilation. `set_trace()` and `clear_trace()` invalidate relevant caches automatically.

### Exercises

- Compare the adjacency tables of an MPS and its Hermitian view.
- Split a multi-core network with `chunk()` and concatenate it again.
- Confirm that `row_priority` and `einsum_default` agree on a small network.
